In [1]:
!pip install torch "transformers>=5" huggingface_hub psutil ipykernel


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import time, statistics
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DTYPE = torch.float32 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()

gen_eos = model.generation_config.eos_token_id
EOS_IDS = set(gen_eos if isinstance(gen_eos, list) else [gen_eos])
GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0)

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

print("loaded", MODEL_ID, DTYPE, "on", DEVICE, "| EOS ids:", EOS_IDS)

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 275.64it/s]


loaded Qwen/Qwen2.5-0.5B-Instruct torch.float32 on cpu | EOS ids: {151643, 151645}


In [14]:
messages = [{"role": "user", "content": "Explain what a KV cache is in two sentences."}]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
MAX_NEW = 40
print("prompt tokens: ", input_ids.shape[1])

@torch.inference_mode()
def greedy_with_cache(input_ids, max_new_tokens):
    # PREFILL: the whole prompt in ONE forward pass -> logits for the next token + the KV cache
    sync()
    t0 = time.perf_counter()
    out = model(input_ids=input_ids, use_cache=True)
    next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    sync()
    prefill_s = time.perf_counter() - t0
    past = out.past_key_values
    generated = [next_id.item()]
    
    # DECODE: one new token per forward pass, reusing the cache
    step_times = []
    for _ in range(max_new_tokens - 1):
        if generated[-1] in EOS_IDS:
            break
        
        sync()
        t = time.perf_counter()
        out = model(input_ids=next_id, past_key_values=past, use_cache=True)
        past = out.past_key_values
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        
        sync()
        step_times.append(time.perf_counter() - t)
        generated.append(next_id.item())
        
    return generated, prefill_s, step_times, past

_ = greedy_with_cache(input_ids, 4)         # warm-up

gen_cache, prefill_s, decode_steps, past = greedy_with_cache(input_ids, MAX_NEW)

# correctness check: our loop must produce exactly what model.generate produces
with torch.inference_mode():
    ref = model.generate(input_ids=input_ids, attention_mask=torch.ones_like(input_ids), max_new_tokens=MAX_NEW, **GREEDY)[0, input_ids.shape[1]:].tolist()
    
print("same tokens as model.generate:", gen_cache == ref)

decode_med = statistics.median(decode_steps)
print(f"\nPREFILL : {input_ids.shape[1]} prompt tokens in {prefill_s*1000:.1f} ms "
      f"-> this is the time-to-first-token (TTFT) on this machine")
print(f"DECODE  : {len(decode_steps)} steps, median {decode_med*1000:.1f} ms/token "
      f"-> {1/decode_med:.1f} tokens/s (inter-token latency)")
print("\nanswer:", tokenizer.decode(gen_cache, skip_special_tokens=True))
    

prompt tokens:  40
same tokens as model.generate: True

PREFILL : 40 prompt tokens in 158.5 ms -> this is the time-to-first-token (TTFT) on this machine
DECODE  : 30 steps, median 60.6 ms/token -> 16.5 tokens/s (inter-token latency)

answer: A KV cache is a type of memory-based data storage system that stores key-value pairs, allowing for quick retrieval of data based on a unique key.


##### Look inside the KV cache

In [15]:
n_layers = len(past.layers)
k0 = past.layers[0].keys

print("cache type        :", type(past).__name__)
print("layers in cache   :", n_layers)
print("keys shape layer 0:", tuple(k0.shape), "= [batch, kv_heads, seq_len, head_dim]")
print("cached seq length :", past.get_seq_length())

actual_bytes = sum(l.keys.numel() * l.keys.element_size() + l.values.numel() * l.values.element_size() for l in past.layers)

cfg = model.config
head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
per_token = 2*cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * k0.element_size()

expected = per_token * past.get_seq_length()

print(f"\nKV bytes per token ({k0.dtype}): {per_token:,}")
print(f"actual cache size : {actual_bytes:,} bytes")
print(f"formula           : {expected:,} bytes  -> match: {actual_bytes == expected}")
print(f"\nNote: kv_heads in the cache = {k0.shape[1]}, not {cfg.num_attention_heads} -> GQA keeps the cache small")


cache type        : DynamicCache
layers in cache   : 24
keys shape layer 0: (1, 2, 70, 64) = [batch, kv_heads, seq_len, head_dim]
cached seq length : 70

KV bytes per token (torch.float32): 24,576
actual cache size : 1,720,320 bytes
formula           : 1,720,320 bytes  -> match: True

Note: kv_heads in the cache = 2, not 14 -> GQA keeps the cache small


##### Turn the cache off

In [18]:
@torch.inference_mode()
def greedy_no_cache(input_ids, max_new_tokens):
    seq = input_ids
    generated, step_times = [], []
    
    for _ in range(max_new_tokens):
        sync()
        t = time.perf_counter()
        logits = model(input_ids=seq, use_cache=False).logits
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        sync()
        
        step_times.append(time.perf_counter() - t)
        generated.append(next_id.item())
        seq = torch.cat([seq, next_id], dim=1)
        if generated[-1] in EOS_IDS:
            break
    return generated, step_times

_ = greedy_no_cache(input_ids, 3)
gen_nocache, nocache_steps = greedy_no_cache(input_ids, MAX_NEW)
print("same tokens as the cached loop:", gen_nocache == gen_cache)

total_cache = prefill_s + sum(decode_steps)
total_nocache = sum(nocache_steps)
med = lambda xs: statistics.median(xs) * 1000
print(f"\nWITH cache   : total {total_cache:.2f}s | decode step ms: first 5 median {med(decode_steps[:5]):.1f}, "
      f"last 5 median {med(decode_steps[-5:]):.1f}")
print(f"WITHOUT cache: total {total_nocache:.2f}s | step ms: first 5 median {med(nocache_steps[:5]):.1f}, "
      f"last 5 median {med(nocache_steps[-5:]):.1f}")
print(f"speed-up from KV cache: {total_nocache/total_cache:.2f}x for {len(gen_cache)} new tokens")

same tokens as the cached loop: True

WITH cache   : total 1.99s | decode step ms: first 5 median 58.7, last 5 median 61.4
WITHOUT cache: total 26.94s | step ms: first 5 median 164.6, last 5 median 1760.6
speed-up from KV cache: 13.57x for 31 new tokens


##### How prefill scales with prompt length

In [19]:
PREFILL_LENGTHS = [32, 128, 512, 1024]
base_ids = tokenizer("The quick brown fox jumps over the lazy dog. " * 400, add_special_tokens=False, return_tensors="pt")["input_ids"].to(DEVICE)

@torch.inference_mode()
def time_prefill(n, repeats=3):
    ids = base_ids[:, :n]
    model(input_ids=ids, use_cache=True)
    times = []
    
    for _ in range(repeats):
        sync()
        t = time.perf_counter()
        model(input_ids=ids, use_cache=True)
        sync()
        times.append(time.perf_counter() - t)
    return statistics.median(times)

print(f"{'prompt tokens':>13} | {'prefill ms':>10} | {'ms per prompt token':>19} | {'prompt tokens/s':>15}")
prefill_table = []

for n in PREFILL_LENGTHS:
    s = time_prefill(n)
    prefill_table.append((n, s))
    print(f"{n:>13} | {s*1000:>10.1f} | {s*1000/n:>19.3f} | {n/s:>15.0f}")


prompt tokens | prefill ms | ms per prompt token | prompt tokens/s
           32 |      139.3 |               4.354 |             230
          128 |     1100.4 |               8.597 |             116
          512 |     8096.5 |              15.813 |              63
         1024 |     3138.9 |               3.065 |             326


##### Decode speed at different context lengths

In [21]:
@torch.inference_mode()
def time_decode_at_context(n, steps=10):
    out = model(input_ids=base_ids[:, :n], use_cache=True) 
    past = out.past_key_values
    next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    times = []
    for _ in range(steps):
        sync()
        t = time.perf_counter()
        out = model(input_ids=next_id, past_key_values=past, use_cache=True)
        sync()
        times.append(time.perf_counter() - t)
        past = out.past_key_values
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    return statistics.median(times)

time_decode_at_context(32, steps=2)
print(f"{'context tokens':>14} | {'decode ms/token':>15} | {'tokens/s':>8}")
for n in PREFILL_LENGTHS:
    s = time_decode_at_context(n)
    print(f"{n:>14} | {s*1000:>15.1f} | {1/s:>8.1f}")

context tokens | decode ms/token | tokens/s
            32 |            65.2 |     15.3
           128 |            68.1 |     14.7
           512 |            70.3 |     14.2
          1024 |            73.0 |     13.7


In [22]:
n_prompt = input_ids.shape[1]
print(f"Model {MODEL_ID} | {DTYPE} | {DEVICE}")
print(f"TTFT (prefill {n_prompt} tokens)      : {prefill_s*1000:.1f} ms")
print(f"Decode median per token        : {decode_med*1000:.1f} ms ({1/decode_med:.1f} tok/s)")
print(f"Cache vs no-cache speed-up     : {total_nocache/total_cache:.2f}x ({len(gen_cache)} tokens)")
big_n, big_s = prefill_table[-1]
print(f"Prefill throughput at {big_n} tok  : {big_n/big_s:.0f} prompt tokens/s "
      f"vs decode {1/decode_med:.1f} new tokens/s")

Model Qwen/Qwen2.5-0.5B-Instruct | torch.float32 | cpu
TTFT (prefill 40 tokens)      : 158.5 ms
Decode median per token        : 60.6 ms (16.5 tok/s)
Cache vs no-cache speed-up     : 13.57x (31 tokens)
Prefill throughput at 1024 tok  : 326 prompt tokens/s vs decode 16.5 new tokens/s
